In [2]:
import os
import re
import glob
import json

In [ ]:


# --- Configuration ---
TEMPLATE_HTML = "demo_interface.html"
OUTPUT_HTML = "demo_interface_updated.html"

MODELS = [
    "AVES-bio",
    "BirdAVES-biox-base",
    "DistilHuBERT",
    "HuBERT",
    "MusicHuBERT",
    "WavLM",
]
SHOTS = [25, 20, 15, 10, 5, 1]

# Display order in UI -> candidate (subfolder, json_filename, json_key) paths
SCORE_SOURCES = {
    "LogME": [
        ("logME", "all_models_logme.json", "overall_logme"),
    ],
    "LFC": [
        ("LFC_score", "all_models_lfc.json", "overall_lfc"),
    ],
    "LGC": [
        ("LGC_linear", "all_models_lgc.json", "overall_lgc"),
    ],
    "LFC_CLS": [
        ("LGC_cls_linear_new", "all_models_lgc_cls_new.json", "overall_lgc_cls_new"),
        ("LGC_cls_linear", "all_models_lgc_cls.json", "overall_lgc_cls"),
    ],
    "LGC-margin": [
        ("SVM_margin_linear_new", "all_models_svm_margin_new.json", "overall_svm_margin_new"),
        ("SVM_margin_linear", "all_models_svm_margin.json", "overall_svm_margin"),
    ],
    "LGC-LOO-margin": [
        ("SVM_loo_margin_linear_new", "all_models_svm_loo_margin_new.json", "overall_loo_margin_new"),
        ("SVM_margin_linear", "all_models_svm_margin.json", "overall_loo_margin"),
    ],
}

root_prefix = ".." if os.path.exists("../model_scores") else "."
SCORES_ROOT = os.path.join(root_prefix, "model_scores")
MODELS_ROOT = os.path.join(root_prefix, "classification_models_trained")


def load_scores_dict() -> dict:
    sc = {}
    for k in SHOTS:
        sc[k] = {}
        shot_dir = os.path.join(SCORES_ROOT, f"model_scores_{k}_shot")
        for display_name, candidates in SCORE_SOURCES.items():
            loaded = False
            for sub_dir, fname, key in candidates:
                fpath = os.path.join(shot_dir, sub_dir, fname)
                if os.path.exists(fpath):
                    with open(fpath, "r") as f:
                        raw = json.load(f)
                    vals = []
                    for m in MODELS:
                        v = raw.get(m, {}).get(key, None)
                        vals.append(round(float(v), 6) if v is not None else None)
                    sc[k][display_name] = vals
                    loaded = True
                    break
            if not loaded:
                raise FileNotFoundError(
                    f"Missing score file for '{display_name}' ({k}-shot) under {shot_dir}"
                )
    return sc


def load_eval_metrics(regime_prefix: str) -> tuple[dict, dict]:
    """
    Finds the evaluation JSON inside trained_models_{regime_prefix}_{k}_shot/Linear/
    and extracts per-model test accuracy (%) and test cross-entropy loss.
    """
    acc_out, loss_out = {}, {}
    for k in SHOTS:
        linear_dir = os.path.join(
            MODELS_ROOT, f"trained_models_{regime_prefix}_{k}_shot", "Linear"
        )
        json_files = [
            p for p in glob.glob(os.path.join(linear_dir, "*.json"))
            if not p.endswith("training_summary.json")
        ]
        if not json_files:
            raise FileNotFoundError(
                f"No test evaluation JSON found in {linear_dir}. "
                f"Ensure test evaluation results are saved there."
            )

        with open(json_files[0], "r") as f:
            raw = json.load(f)

        # Unwrap if nested under a top-level results key
        if "results" in raw and isinstance(raw["results"], dict):
            raw = raw["results"]

        accs, losses = [], []
        for m in MODELS:
            entry = raw[m]
            acc_val = next(
                entry[key] for key in ("test_accuracy", "test_acc", "accuracy", "acc") if key in entry
            )
            loss_val = next(
                entry[key] for key in ("test_loss", "loss", "ce_loss") if key in entry
            )
            # Convert fraction [0, 1] to percentage if needed
            acc_pct = acc_val * 100.0 if acc_val <= 1.0 else float(acc_val)
            accs.append(round(acc_pct, 2))
            losses.append(round(float(loss_val), 4))

        acc_out[k] = accs
        loss_out[k] = losses

    return acc_out, loss_out


def build_html():
    with open(TEMPLATE_HTML, "r", encoding="utf-8") as f:
        html = f.read()

    sc_data = load_scores_dict()
    ac_g1, loss_g1 = load_eval_metrics("lambda_tuned")
    ac_g2, loss_g2 = load_eval_metrics("epoch_tuned")

    ac_data = {"G1": ac_g1, "G2": ac_g2}
    loss_data = {"G1": loss_g1, "G2": loss_g2}

    ts_list = list(SCORE_SOURCES.keys())
    ms_list = ["Pearson", "Kendall", "Spearman"]

    # 1. Replace constants TS, MS, SHOTS, SC, AC, LOSS
    html = re.sub(
        r'const TS=\[.*?\],MS=\[.*?\],SHOTS=\[.*?\];',
        f'const TS={json.dumps(ts_list)},MS={json.dumps(ms_list)},SHOTS={json.dumps(SHOTS)};',
        html,
    )
    html = re.sub(
        r'const SC=\{.*?\};\n\[10,5,1\]\.forEach\(.*?\);',
        f'const SC={json.dumps(sc_data)};',
        html,
        flags=re.DOTALL,
    )
    html = re.sub(
        r'const AC=\{.*?\};\nconst LOSS=\{.*?\};',
        f'const AC={json.dumps(ac_data)};\nconst LOSS={json.dumps(loss_data)};',
        html,
        flags=re.DOTALL,
    )

    # 2. Set default Rank metric radio selection to Pearson
    html = html.replace(
        'grp("r","Rank metric",MS,"radio",["Spearman"])',
        'grp("r","Rank metric",MS,"radio",["Pearson"])',
    )

    # 3. Add null/NaN handling for 1-shot LGC-LOO-margin across stats, tables, and SVGs
    old_corr = 'function corr(m,x,y){const f=FN[m],r=f(x,y);if(isNaN(r))return{r:NaN,p:NaN};let h=0,t=0;perms(y).forEach(p=>{t++;if(Math.abs(f(x,p))>=Math.abs(r)-1e-9)h++});return{r,p:h/t}}'
    new_corr = 'function corr(m,x,y){const idx=x.map((_,i)=>i).filter(i=>x[i]!=null&&!isNaN(x[i])&&y[i]!=null&&!isNaN(y[i]));if(idx.length<3)return{r:NaN,p:NaN};const xv=idx.map(i=>x[i]),yv=idx.map(i=>y[i]),f=FN[m],r=f(xv,yv);if(isNaN(r))return{r:NaN,p:NaN};let h=0,t=0;perms(yv).forEach(p=>{t++;if(Math.abs(f(xv,p))>=Math.abs(r)-1e-9)h++});return{r,p:h/t}}'
    html = html.replace(old_corr, new_corr)

    old_bars = 'function bars(t,sh,mi){const v=mi.map(i=>SC[sh][t][i]),lo=Math.min(...v),hi=Math.max(...v);\nreturn`<div class="card"><h3>${t}</h3>`+mi.map((i,k)=>`<div class="bar"><span>${SH[i]}</span><span><i style="width:${hi==lo?100:12+88*(v[k]-lo)/(hi-lo)}%;background:${COL[i]}"></i></span><b>${v[k].toFixed(4)}</b></div>`).join("")+`<small style="color:var(--mut)">bar length scaled within this score</small></div>`}'
    new_bars = 'function bars(t,sh,mi){const v=mi.map(i=>SC[sh][t][i]),valid=v.filter(x=>x!=null&&!isNaN(x));if(!valid.length)return`<div class="card"><h3>${t}</h3><small style="color:var(--mut)">N/A (requires K ≥ 2)</small></div>`;const lo=Math.min(...valid),hi=Math.max(...valid);\nreturn`<div class="card"><h3>${t}</h3>`+mi.map((i,k)=>`<div class="bar"><span>${SH[i]}</span><span><i style="width:${v[k]==null?0:(hi==lo?100:12+88*(v[k]-lo)/(hi-lo))}%;background:${COL[i]}"></i></span><b>${v[k]==null?"–":v[k].toFixed(4)}</b></div>`).join("")+`<small style="color:var(--mut)">bar length scaled within this score</small></div>`}'
    html = html.replace(old_bars, new_bars)

    old_sc_rank = 'const scRank=(t,sh,mi)=>rkI(mi.map(i=>SC[sh][t][i]));'
    new_sc_rank = 'const scRank=(t,sh,mi)=>{const v=mi.map(i=>SC[sh][t][i]);return v.some(x=>x==null||isNaN(x))?mi.map(()=>null):rkI(v)};'
    html = html.replace(old_sc_rank, new_sc_rank)

    old_scatter = 'function scatter(t,sh,g,mi){const x=mi.map(i=>SC[sh][t][i]),y=mi.map(i=>AC[g][sh][i]),W=440,H=270,L=50,R=24,T=22,B=40,x0=Math.min(...x),x1=Math.max(...x),y0=Math.min(...y),y1=Math.max(...y),'
    new_scatter = 'function scatter(t,sh,g,mi){const rawX=mi.map(i=>SC[sh][t][i]);if(rawX.some(v=>v==null||isNaN(v)))return`<div class="card"><h3>${t} <small>vs accuracy</small></h3><small style="color:var(--mut)">N/A (requires K ≥ 2)</small></div>`;const x=rawX,y=mi.map(i=>AC[g][sh][i]),W=440,H=270,L=50,R=24,T=22,B=40,x0=Math.min(...x),x1=Math.max(...x),y0=Math.min(...y),y1=Math.max(...y),'
    html = html.replace(old_scatter, new_scatter)

    old_rank_lines = 'mi.forEach((i,j)=>{s+=`<polyline fill="none" stroke="${COL[i]}" stroke-width="2.5" opacity=".9" points="${cols.map((c,k)=>px(k)+","+py(c[j])).join(" ")}"/>`+cols.map((c,k)=>`<circle cx="${px(k)}" cy="${py(c[j])}" r="10" fill="${COL[i]}"/><text x="${px(k)}" y="${py(c[j])+4}" text-anchor="middle" style="fill:#0e1512;font-weight:700">${nm(c[j])}</text>`).join("")+'
    new_rank_lines = 'mi.forEach((i,j)=>{const validPts=cols.map((c,k)=>c[j]==null?null:px(k)+","+py(c[j])).filter(Boolean);s+=`<polyline fill="none" stroke="${COL[i]}" stroke-width="2.5" opacity=".9" points="${validPts.join(" ")}"/>`+cols.map((c,k)=>c[j]==null?"":`<circle cx="${px(k)}" cy="${py(c[j])}" r="10" fill="${COL[i]}"/><text x="${px(k)}" y="${py(c[j])+4}" text-anchor="middle" style="fill:#0e1512;font-weight:700">${nm(c[j])}</text>`).join("")+'
    html = html.replace(old_rank_lines, new_rank_lines)

    old_ranks = '${cols.map(c=>`<td class="c${Math.min(2,Math.round(Math.abs(c[k]-gt[k])))}">${nm(c[k])}</td>`).join("")}'
    new_ranks = '${cols.map(c=>c[k]==null?`<td>–</td>`:`<td class="c${Math.min(2,Math.round(Math.abs(c[k]-gt[k])))}">${nm(c[k])}</td>`).join("")}'
    html = html.replace(old_ranks, new_ranks)

    old_sweep = 'const C=["#e8b84a","#6fcf97","#7fb3e8","#c79af0"];'
    new_sweep = 'const C=["#e8b84a","#6fcf97","#7fb3e8","#c79af0","#e5735c","#5fd0c4"];'
    html = html.replace(old_sweep, new_sweep)
    html = html.replace("C[j%4]", "C[j%6]")

    old_sweep_pts = 'const pts=xs.map((sh,k)=>[px(k),py(FN[m](mi.map(i=>SC[sh][t][i]),mi.map(i=>AC[g][sh][i])))]);'
    new_sweep_pts = 'const pts=xs.map((sh,k)=>{const xv=mi.map(i=>SC[sh][t][i]);if(xv.some(v=>v==null||isNaN(v)))return null;return[px(k),py(FN[m](xv,mi.map(i=>AC[g][sh][i])))]}).filter(Boolean);'
    html = html.replace(old_sweep_pts, new_sweep_pts)

    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Successfully generated {OUTPUT_HTML} with all 6 shot settings and 6 transferability scores.")


if __name__ == "__main__":
    build_html()

SyntaxError: invalid syntax (4201402480.py, line 185)